In [ ]:
# 파이단틱 이용하기
# 입력된 데이터의 유효성과 형식을 검증하고 특정 데이터 형식으로 명확하게 표현할때 사용하는 라이브러리 
# 주가 정보를 가져오는 종목(티커: ticker) 기간(period)를 매개변수로 받는다.
# 이 매개변수의 형식을 파이단틱 베이스모델(BaseModel)과 필드(Field)를 아용해 더욱 명확하게 정의할 수 있다.
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 잘 지냈습니다! 당신은 어떻게 지내세요? 도움이 필요하신 사항이 있으면 말씀해 주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 12, 'total_tokens': 38, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_255abcd69b', 'id': 'chatcmpl-DWYBPA49ErTVuhWyTOF0RdScnagkM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da89b-a45d-7820-9618-54c5dd123269-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 26, 'total_tokens': 38, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [3]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_baaa53d70d', 'id': 'chatcmpl-DWYEAuN6FG3AjKdgZItjsgvMjPOrI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da89e-4318-7a40-a163-ddb63b796102-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_KLOqjxLcr2QpDiNQiwImaRel', 'type': 'tool_call'}],

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-04-20 11:00:47 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_baaa53d70d', 'id': 'chatcmpl-DWYEAuN6FG3AjKdgZItjsgvMjPOrI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da89e-4318-7a40-a163-ddb63b796102-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_KLOqjxLcr2QpDiNQiwImaRel', 'type': 'tool_call'}

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='부산의 현재 시각은 2026년 4월 20일 11시 00분 47초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 192, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DWYEW3t7dLUvmsMqfihEBi5ncSHrx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da89e-9916-76e2-b7d6-20f0ff76005f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 29, 'total_tokens': 221, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
# 파이단틱 코드 추가
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [9]:
!pip install yfinance
import yfinance as yf 

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 22.2 MB/s  0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15702 sha256=a63d2412ffc01010fd51da1ae144a60e24de7ca7f6ef4775a380c76b20a7467b
  Stored in directory: c:\users\mbc320\appdata\local\pip\cache\wheels\cc\bd\6f\664d62c99327abeef7d86489e6631cbf45b56fbf7ef1d6ef00
Successfully built multitasking

   ---------------------------------------- 0/8 [peewee]
   ---------------------------------------- 0/8 [peewee]
   ---------------------------------------- 0/8 [peewee]
   ---------- ----------------

In [10]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 283, 'total_tokens': 310, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fe7a5277a8', 'id': 'chatcmpl-DWYGjEVbTFb6qiFTEvXFnvVCMLclE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019da8a0-af6f-7740-bb78-4c9d0863e926-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_cVIovKAsqwpcQmNYKWruC7X8', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 283, 'output_tokens': 27, 'total_tokens': 310, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

In [ ]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)


# content='
# | Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n
# |:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n
# | 2026-03-18 00:00:00-04:00 | 399    | 403.07 | 392.31 |  392.78 | 5.08531e+07 |           0 |              0 |\n
# | 2026-03-19 00:00:00-04:00 | 387.27 | 387.27 | 378.73 |  380.3  | 6.70783e+07 |           0 |              0 |\n
# | 2026-03-20 00:00:00-04:00 | 379.85 | 379.89 | 364.46 |  367.96 | 7.86286e+07 |           0 |              0 |\n
# | 2026-03-23 00:00:00-04:00 | 373.09 | 385.33 | 372.73 |  380.85 | 7.4606e+07  |           0 |              0 |\n
# | 2026-03-24 00:00:00-04:00 | 376.56 | 387.48 | 376.31 |  383.03 | 6.00049e+07 |           0 |              0 |\n
# | 2026-03-25 00:00:00-04:00 | 389.99 | 396.23 | 385.01 |  385.95 | 5.51573e+07 |           0 |              0 |\n
# | 2026-03-26 00:00:00-04:00 | 381.6  | 384.44 | 371.87 |  372.11 | 5.55229e+07 |           0 |              0 |\n
# | 2026-03-27 00:00:00-04:00 | 369.69 | 369.86 | 359.47 |  361.83 | 6.20657e+07 |           0 |              0 |\n
# | 2026-03-30 00:00:00-04:00 | 365.86 | 367.29 | 352.14 |  355.28 | 6.79544e+07 |           0 |              0 |\n
# | 2026-03-31 00:00:00-04:00 | 361.51 | 373.33 | 361    |  371.75 | 7.55349e+07 |           0 |              0 |\n
# | 2026-04-01 00:00:00-04:00 | 378.63 | 383.14 | 374.08 |  381.26 | 5.86838e+07 |           0 |              0 |\n
# | 2026-04-02 00:00:00-04:00 | 364.2  | 370.28 | 359.03 |  360.59 | 8.30312e+07 |           0 |              0 |\n
# | 2026-04-06 00:00:00-04:00 | 362.59 | 367.72 | 346.64 |  352.82 | 7.76976e+07 |           0 |              0 |\n
# | 2026-04-07 00:00:00-04:00 | 346.44 | 348.02 | 337.24 |  346.65 | 7.45154e+07 |           0 |              0 |\n
# | 2026-04-08 00:00:00-04:00 | 363.79 | 364.5  | 339.67 |  343.25 | 7.88386e+07 |           0 |              0 |\n
# | 2026-04-09 00:00:00-04:00 | 343.15 | 348.88 | 337.25 |  345.62 | 6.2164e+07  |           0 |              0 |\n
# | 2026-04-10 00:00:00-04:00 | 346.29 | 350.36 | 342.74 |  348.95 | 5.1336e+07  |           0 |              0 |\n
# | 2026-04-13 00:00:00-04:00 | 350.07 | 356.35 | 348.57 |  352.42 | 5.36175e+07 |           0 |              0 |\n
# | 2026-04-14 00:00:00-04:00 | 357.67 | 367.63 | 354.77 |  364.2  | 5.99796e+07 |           0 |              0 |\n
# | 2026-04-15 00:00:00-04:00 | 366.83 | 394.65 | 362.5  |  391.95 | 1.1381e+08  |           0 |              0 |\n
# | 2026-04-16 00:00:00-04:00 | 393.81 | 394.06 | 381.8  |  388.9  | 6.35151e+07 |           0 |              0 |\n
# | 2026-04-17 00:00:00-04:00 | 395.92 | 409.28 | 391.65 |  400.62 | 9.03884e+07 |           0 |              0 |
# ' name='get_yf_stock_history' tool_call_id='call_cVIovKAsqwpcQmNYKWruC7X8'


{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-03-18 00:00:00-04:00 | 399    | 403.07 | 392.31 |  392.78 | 5.08531e+07 |           0 |              0 |\n| 2026-03-19 00:00:00-04:00 | 387.27 | 387.27 | 378.73 |  380.3  | 6.70783e+07 |           0 |              0 |\n| 2026-03-20 00:00:00-04:00 | 379.85 | 379.89 | 364.46 |  367.96 | 7.86286e+07 |           0 |              0 |\n| 2026-03-23 00:00:00-04:00 | 373.09 | 385.33 | 372.73 |  380.85 | 7.4606e+07  |           0 |              0 |\n| 2026-03-24 00:00:00-04:00 | 376.56 | 387.48 | 376.31 |  383.03 | 6.00049e+07 |           0 |              0 |\n| 2026-03-25 00:00:00-04:00 | 389.99 | 396.23 | 385.01 |  385.95 | 5.51573e+07 |           0 |              0 |\n| 2026-03-26 00:00:00-04:0

In [12]:
llm_with_tools.invoke(messages)

AIMessage(content='한 달 전인 2026년 3월 18일의 테슬라 주가는 392.78달러였고, 현재(2026년 4월 20일) 주가는 400.62달러입니다. 따라서 테슬라의 주가는 한 달 전에 비해 올랐습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 1640, 'total_tokens': 1709, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fe7a5277a8', 'id': 'chatcmpl-DWYHC7RprGWcDZl0NlBLT4QUeIChn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da8a1-24c8-76f3-98b3-3af4698e1826-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1640, 'output_tokens': 69, 'total_tokens': 1709, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})